# Analysis of Single Neuron Processing Suite

This notebook visualizes and analyzes the impact of various image processing and meshing parameters on the resulting tetrahedral meshes of a single neuron.

## Experimental Design
The results were generated by systematically varying three groups of parameters:
1. **Group A: Resolution & Scale** - Tests the effect of `mip` (resolution level), `dx` (voxel size), and `env` (envelope size).
2. **Group B: Morphological Radius** - Tests the effect of the dilation/erosion radius on the cell volume and connectivity.
3. **Group C: Smoothing Strength** - Tests the effect of smoothing iterations and radius on surface regularity.

In [ ]:
import os
import re
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown

RESULTS_DIR = Path("../emimesh/results/single_neuron_processing/results")

def parse_folder_name(name):
    """
    Parses folder name: mip{mip}_dx{dx}_rmis10000_dil{dil}_sm{it}{rad}_er{ero}_env{env}
    """
    pattern = r"mip(\d+)_dx(\d+)_rmis10000_dil(\d+)_sm(\d+)(\d+)_er(\d+)_env(\d+)"
    match = re.match(pattern, name)
    if match:
        return {
            "mip": int(match.group(1)),
            "dx": int(match.group(2)),
            "dil": int(match.group(3)),
            "sm_it": int(match.group(4)),
            "sm_rad": int(match.group(5)),
            "ero": int(match.group(6)),
            "env": int(match.group(7)),
            "name": name
        }
    return None

def get_results_metadata():
    metadata = []
    if not RESULTS_DIR.exists():
        print(f"Error: Results directory {RESULTS_DIR} not found.")
        return metadata
    
    for folder in RESULTS_DIR.iterdir():
        if folder.is_dir():
            params = parse_folder_name(folder.name)
            if params:
                img_path = folder / "meshes" / "mesh.png"
                params["img_path"] = img_path if img_path.exists() else None
                metadata.append(params)
    return metadata

all_results = get_results_metadata()
print(f"Found {len(all_results)} processed configurations.")

## Group A: Resolution & Scale

**Constants:** `dil=2, ero=2, sm_it=1, sm_rad=2`

This sweep evaluates how the choice of resolution (`mip`) and voxel size (`dx`) affect the reconstructed neuron. The envelope size (`env`) is also varied to ensure the mesh adequately captures the neuron's extent without excessive overhead.

In [ ]:
group_a = [r for r in all_results if r['dil'] == 2 and r['ero'] == 2 and r['sm_it'] == 1 and r['sm_rad'] == 2]
# Sort by mip and dx for consistent layout
group_a.sort(key=lambda x: (x['mip'], x['dx']))

def plot_group(results, title):
    if not results:
        print("No results found for this group.")
        return
    
    n = len(results)
    cols = 3
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    axes = axes.flatten() if n > 1 else [axes]
    
    for i, res in enumerate(results):
        ax = axes[i]
        if res['img_path']:
            img = plt.imread(res['img_path'])
            ax.imshow(img)
            label = f"mip={res['mip']}, dx={res['dx']}, env={res['env']}"
            ax.set_title(label, fontsize=10)
        else:
            ax.text(0.5, 0.5, "Mesh Failed", ha='center', va='center')
            ax.set_title(res['name'], fontsize=8)
        ax.axis('off')
    
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

plot_group(group_a, "Group A: Resolution & Scale")

## Group B: Morphological Radius

**Constants:** `mip=4, dx=100, env=50, sm_it=1, sm_rad=2`

This sweep examines the impact of the dilation and erosion radii. Morphological operations are used to fill small gaps and remove noise; however, excessive radii can lead to the merging of distinct neurites or the loss of fine structural detail.

In [ ]:
group_b = [r for r in all_results if r['mip'] == 4 and r['dx'] == 100 and r['env'] == 50 and r['sm_it'] == 1 and r['sm_rad'] == 2]
group_b.sort(key=lambda x: x['dil'])

def plot_group_b(results, title):
    if not results:
        print("No results found for this group.")
        return
    
    n = len(results)
    cols = 3
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    axes = axes.flatten() if n > 1 else [axes]
    
    for i, res in enumerate(results):
        ax = axes[i]
        if res['img_path']:
            img = plt.imread(res['img_path'])
            ax.imshow(img)
            label = f"rad={res['dil']}"
            ax.set_title(label, fontsize=10)
        else:
            ax.text(0.5, 0.5, "Mesh Failed", ha='center', va='center')
            ax.set_title(res['name'], fontsize=8)
        ax.axis('off')
    
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

plot_group_b(group_b, "Group B: Morphological Radius")

## Group C: Smoothing Strength

**Constants:** `mip=4, dx=100, env=50, dil=2, ero=2`

This sweep tests the effect of the smoothing filter. Smoothing is critical for reducing the 'staircase' effect of voxelization, which can lead to better quality tetrahedral meshes and more accurate numerical simulations.

In [ ]:
group_c = [r for r in all_results if r['mip'] == 4 and r['dx'] == 100 and r['env'] == 50 and r['dil'] == 2 and r['ero'] == 2]
group_c.sort(key=lambda x: (x['sm_it'], x['sm_rad']))

def plot_group_c(results, title):
    if not results:
        print("No results found for this group.")
        return
    
    n = len(results)
    cols = 2
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(12, 5 * rows))
    axes = axes.flatten() if n > 1 else [axes]
    
    for i, res in enumerate(results):
        ax = axes[i]
        if res['img_path']:
            img = plt.imread(res['img_path'])
            ax.imshow(img)
            label = f"it={res['sm_it']}, rad={res['sm_rad']}"
            ax.set_title(label, fontsize=10)
        else:
            ax.text(0.5, 0.5, "Mesh Failed", ha='center', va='center')
            ax.set_title(res['name'], fontsize=8)
        ax.axis('off')
    
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

plot_group_c(group_c, "Group C: Smoothing Strength")

## Summary and Observations

Based on the visualizations above, we can observe:
1. **Resolution**: Higher `mip` and `dx` values lead to coarser meshes, potentially losing fine-grained morphological details but reducing computational cost.
2. **Morphology**: There is a trade-off between connectivity and fidelity. Low radii may leave gaps in the segmentation, while high radii merge nearby neurites.
3. **Smoothing**: Increased smoothing significantly reduces surface noise, which is essential for the stability of the tetrahedral meshing process.

Conclusions: 
- Smoothing iterations 1 radius 2
- Dilation/erosion raddius 2 (or maybe 1)
- Resolution?